# Model Evaluation

Runs NER, tags, and summary on 2 fixed eval docs for every model in `MODELS`.
Output is displayed inline for qualitative grading. Results are saved per model under `data/results/<model>/`.

In [1]:
MODELS = [
    # 'qwen2.5:7b',
    # 'llama3.2:latest',
    # 'qwen3:14b',
    'llama3.1:8b',
    # 'qwen3:8b',
]

EVAL_DOCS = [
    'medium_climate_blog_paris_agreement_en.md',
    'medium_klimaat_blog_parijs_akkoord_nl.md',
]

In [2]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
sys.path.insert(0, '../src')

from pathlib import Path

from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.tasks import run_ner, run_summary, run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

EVAL_DOCS_DIR = Path('../data/synthetic_docs')
backend = config['run']['backend']
print(f'backend={backend}  models={MODELS}')

backend=ollama  models=['llama3.1:8b']


In [3]:
all_results = {}

for model in MODELS:
    config[backend]['model'] = model
    print(f'\n--- {model} ---')
    results = []
    for doc in EVAL_DOCS:
        post = load_note(str(EVAL_DOCS_DIR / doc))
        row = {'file': doc}
        for task, fn in [('ner', run_ner), ('tags', run_tags), ('summary', run_summary)]:
            t0 = time.perf_counter()
            try:
                row[task] = fn(backend, post.content, config)
            except Exception as e:
                row[task] = f'ERROR: {e}'
            row[f'{task}_s'] = round(time.perf_counter() - t0, 1)
            print(f'  {doc}  {task}  ({row[f"{task}_s"]}s)')
        results.append(row)
    all_results[model] = results

print('\nAll models done')


--- llama3.1:8b ---
  medium_climate_blog_paris_agreement_en.md  ner  (21.9s)
  medium_climate_blog_paris_agreement_en.md  tags  (4.7s)
  medium_climate_blog_paris_agreement_en.md  summary  (7.5s)
  medium_klimaat_blog_parijs_akkoord_nl.md  ner  (27.5s)
  medium_klimaat_blog_parijs_akkoord_nl.md  tags  (4.9s)
  medium_klimaat_blog_parijs_akkoord_nl.md  summary  (6.0s)

All models done


In [ ]:
records = []
for model, results in all_results.items():
    for r in results:
        for task in ('ner', 'tags', 'summary'):
            payload = r[task]
            is_error = isinstance(payload, str) and payload.startswith('ERROR:')
            records.append({
                'model': model,
                'task': task,
                'doc': r['file'],
                'elapsed_s': r[f'{task}_s'],
                'result': payload if not is_error else None,
                'error': payload if is_error else None,
            })

out_path = Path('../data/results/model_eval.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(records, ensure_ascii=False, indent=2))
print(f'Saved {len(records)} records to {out_path}')